# 제품 이상여부 판별 프로젝트


## 1. 데이터 불러오기


### 필수 라이브러리


In [4]:
!pip install xgboost
!pip install catboost
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 MB 14.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 66.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 9.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


### 데이터 읽어오기


In [5]:
ROOT_DIR = "data"
RANDOM_STATE = 110

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
list(train_data)
print("The total number of colums: " + str(len(train_data.columns)))

The total number of colums: 464


In [6]:
na_cal = []

for i in train_data.columns:
    if (train_data[i].isna().sum()) > 0:
        na_cal.append(i)

print("Total number of columns with NA: " + str(len(na_cal)))

train_data[na_cal] = train_data[na_cal].fillna(0)

Total number of columns with NA: 286


In [7]:
one_val_duplicated = []
for i in train_data.columns:
    if (train_data[i].nunique()) == 1:
        one_val_duplicated.append(i)
        
print("Total number of columns with only one val: " + str(len(one_val_duplicated)))

Total number of columns with only one val: 313


In [8]:
# "Number of unique entries = Num rows" ==> "Unique value for every row"
num_rows = len(train_data)
unique_every_row = []
for i in train_data.columns:
    if (train_data[i].value_counts().size == num_rows):
        unique_every_row.append(i)
        
print("Total number of columns with unique values for every row: " + str(len(unique_every_row)))

Total number of columns with unique values for every row: 0


In [9]:
multiple_types = []
for i in train_data.columns:
    if (len(set(train_data[i].apply(type))) > 1):
        multiple_types.append(i)

print(multiple_types)
print("Total number of columns with multiple datatypes: " + str(len(multiple_types)))

['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam', 'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave', 'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2']
Total number of columns with multiple datatypes: 8


In [ ]:
# Columns with "OK" and numbers are mixed --> to be processed ("OK" to NaN):
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2

# Columns with "OK" and NaN are mixed:
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam
#     GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave
#     GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2

In [11]:
ok_2_nan = ["HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam", 
            "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1", 
            "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2"]

for i in ok_2_nan:
    train_data[i] = train_data[i].replace('OK', np.nan)
    train_data[i] = train_data[i].astype(float)

### 언더 샘플링


데이타 불균형을 해결하기 위해 언더 샘플링을 진행합니다.


In [12]:
normal_ratio = 1.0  # 1.0 means 1:1 ratio

df_normal = train_data[train_data["target"] == "Normal"]
df_abnormal = train_data[train_data["target"] == "AbNormal"]

num_normal = len(df_normal)
num_abnormal = len(df_abnormal)
print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}")

df_normal = df_normal.sample(n=int(num_abnormal * normal_ratio), replace=False, random_state=RANDOM_STATE)
df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
df_concat.value_counts("target")

  Total: Normal: 38156, AbNormal: 2350


target
AbNormal    2350
Normal      2350
Name: count, dtype: int64

### 데이터 분할


In [13]:
df_train, df_val = train_test_split(
    df_concat,
    test_size=0.3,
    stratify=df_concat["target"],
    random_state=RANDOM_STATE,
)


def print_stats(df: pd.DataFrame):
    num_normal = len(df[df["target"] == "Normal"])
    num_abnormal = len(df[df["target"] == "AbNormal"])

    print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}" + f" ratio: {num_abnormal/num_normal}")


# Print statistics
print(f"  \tAbnormal\tNormal")
print_stats(df_train)
print_stats(df_val)

  	Abnormal	Normal
  Total: Normal: 1645, AbNormal: 1645 ratio: 1.0
  Total: Normal: 705, AbNormal: 705 ratio: 1.0


## 3. 모델 학습


### 모델 정의


In [14]:
features = []

for col in df_train.columns:
    try:
        df_train[col] = df_train[col].astype(int)
        features.append(col)
    except:
        continue

train_x = df_train[features]
train_y = df_train["target"]

In [15]:
features = []

for col in df_val.columns:
    try:
        df_val[col] = df_val[col].astype(int)
        features.append(col)
    except:
        continue

test_x = df_val[features]
test_y = df_val["target"]

### 모델 학습


In [16]:
le = LabelEncoder()
encoded_y = le.fit_transform(train_y)
encoded_y_test = le.fit_transform(test_y)
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))

# Print the mapping
print("Label to Integer Mapping:", label_mapping)

Label to Integer Mapping: {'AbNormal': 0, 'Normal': 1}


In [ ]:
# xgb_model = XGBClassifier(random_state=RANDOM_STATE)

# params = {'max_depth' : [5, 7, 10, 15, 20], 
#           'min_child_weight' : [1, 3, 5, 8, 10], 
#           'colsample_bytree' : [0.3, 0.5, 0.75],
#           'learning_rate' : [0.1, 0.15, 0.2],
#           'subsample' : [0.5, 0.75, 1],
#           'colsample_bytree' : [0.5, 0.75, 1]}

# gridcv = GridSearchCV(xgb_model, param_grid=params, cv=5)

# gridcv.fit(train_x, encoded_y)
# print(gridcv.best_params_)

In [17]:
pre_cal_xgbc = XGBClassifier(n_estimators=300, 
                     learning_rate=0.01, 
                     max_depth=3, 
                     min_child_weight=4,
                     subsample=1,
                     colsample_bytree=0.8, 
                     random_state=RANDOM_STATE,
                     n_jobs=-1,
                     gamma=0.3,
                     eval_metric='logloss')

In [18]:
xgbc = CalibratedClassifierCV(pre_cal_xgbc, method='isotonic', cv=5)
xgbc.fit(train_x, encoded_y)

CalibratedClassifierCV(cv=5,
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.8,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric='logloss',
                                               feature_types=None, gamma=0.3,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.01, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=3,
                                               max_leaves=None,
                                               min_child_weight=4, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=300, n_jobs=-1,
                                               num_parallel_tree=None,
                                               random_state=110, ...),
                       method='isotonic')

In [ ]:
# params = { 'n_estimators' : [10, 30, 50, 80, 100, 150],
#            'max_depth' : [6, 8, 10, 12, 15, 20],
#            'min_samples_leaf' : [8, 12, 18, 20],
#            'min_samples_split' : [8, 16, 20, 25],
#            'n_jobs' : [-1],
#            'random_state' : [RANDOM_STATE],
#             }

# # RandomForestClassifier 객체 생성 후 GridSearchCV 수행
# rf_clf = RandomForestClassifier(random_state = 0, n_jobs = -1)
# grid_cv = GridSearchCV(rf_clf, param_grid = params, cv = 3, n_jobs = -1)
# grid_cv.fit(train_x, train_y)

# print('최적 하이퍼 파라미터: ', grid_cv.best_params_)
# print('최고 예측 정확도: {:.4f}'.format(grid_cv.best_score_))

In [19]:
# 모델 학습 (rf)
pre_cal_rf = RandomForestClassifier(n_estimators = 30, 
                                               max_depth = 15, 
                                               min_samples_leaf = 8,
                                               min_samples_split = 20,
                                               n_jobs = -1,
                                               random_state = RANDOM_STATE).fit(train_x, train_y)

In [20]:
rfc = CalibratedClassifierCV(pre_cal_rf, method='isotonic', cv=5)
rfc.fit(train_x, train_y)


CalibratedClassifierCV(cv=5,
                       estimator=RandomForestClassifier(max_depth=15,
                                                        min_samples_leaf=8,
                                                        min_samples_split=20,
                                                        n_estimators=30,
                                                        n_jobs=-1,
                                                        random_state=110),
                       method='isotonic')

In [21]:
# 모델 학습 (SVM)
pre_cal_svc = SVC(probability=True, 
                  random_state=RANDOM_STATE, 
                  class_weight='balanced')


In [22]:
svc = CalibratedClassifierCV(pre_cal_svc, method='sigmoid', cv=7)
svc.fit(train_x, train_y)


CalibratedClassifierCV(cv=7,
                       estimator=SVC(class_weight='balanced', probability=True,
                                     random_state=110))

In [23]:
# 모델 학습 (CatBoost)
pre_cal_catboost = CatBoostClassifier(iterations=300, learning_rate=0.01, depth=5, random_state=RANDOM_STATE, verbose=0)

In [24]:
catboost = CalibratedClassifierCV(pre_cal_catboost, method='isotonic', cv=5)
catboost.fit(train_x, train_y)

CalibratedClassifierCV(cv=5,
                       estimator=<catboost.core.CatBoostClassifier object at 0x7f6055da0130>,
                       method='isotonic')

In [25]:
# Voting Ensemble
model = VotingClassifier(
    estimators = [
        ('Random Forest Classifier', rfc), 
        ('XGBoosting Classifier', xgbc), 
        ('svm', svc),
        ('catboost', catboost)],
    voting='soft', 
    weights=[5, 2, 1, 3])

model.fit(train_x, train_y)

VotingClassifier(estimators=[('Random Forest Classifier',
                              CalibratedClassifierCV(cv=5,
                                                     estimator=RandomForestClassifier(max_depth=15,
                                                                                      min_samples_leaf=8,
                                                                                      min_samples_split=20,
                                                                                      n_estimators=30,
                                                                                      n_jobs=-1,
                                                                                      random_state=110),
                                                     method='isotonic')),
                             ('XGBoosting Classifier',
                              CalibratedClassifierCV(cv=5,
                                                     estimator=XGBClassifier(base_score=None,
                                                                             booster=None,
                                                                             callbacks=No...
                                                                             n_estimators=300,
                                                                             n_jobs=-1,
                                                                             num_parallel_tree=None,
                                                                             random_state=110, ...),
                                                     method='isotonic')),
                             ('svm',
                              CalibratedClassifierCV(cv=7,
                                                     estimator=SVC(class_weight='balanced',
                                                                   probability=True,
                                                                   random_state=110))),
                             ('catboost',
                              CalibratedClassifierCV(cv=5,
                                                     estimator=<catboost.core.CatBoostClassifier object at 0x7f6055da0130>,
                                                     method='isotonic'))],
                 voting='soft', weights=[5, 2, 1, 3])

In [26]:
voting_res = model.predict(test_x)
encoded_voting_res = le.fit_transform(voting_res)
print(f1_score(encoded_y_test, encoded_voting_res))


0.6013793103448276


## 4. 제출하기


### 테스트 데이터 예측


테스트 데이터 불러오기


In [27]:
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

In [28]:
df_test_x = test_data[features]

for col in df_test_x.columns:
    try:
        df_test_x.loc[:, col] = df_test_x[col].astype(int)
    except:
        continue

In [30]:
# NaN 값을 0으로 대체
df_test_x = df_test_x.fillna(0)

test_pred = model.predict(df_test_x)
test_pred

array(['AbNormal', 'Normal', 'AbNormal', ..., 'Normal', 'AbNormal',
       'Normal'], dtype=object)

### 제출 파일 작성


In [32]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
df_sub = pd.read_csv("submission.csv")
df_sub["target"] = test_pred

# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)

**우측 상단의 제출 버튼을 클릭해 결과를 확인하세요**
